# CI2604P: Smoker Status Prediction Challenge

Binary classification (`smoking`: 1 = smoker, 0 = non-smoker) using biological/physical health metrics.

**Pipeline:**
1. Load train/test
2. Feature engineering (BMI, ratios, composite health indices)
3. 5-fold Stratified CV with LightGBM, XGBoost, CatBoost
4. Blend out-of-fold + test predictions (weight-optimized ensemble)
5. Report CV AUC, write `submission.csv`


## 1. Imports

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

RANDOM_STATE = 42
N_SPLITS = 5

## 2. Load Data

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


TARGET = "smoking"
ID_COL = "id"

train_ids = train[ID_COL].copy()
test_ids = test[ID_COL].copy()
y = train[TARGET].astype(int).copy()

train = train.drop(columns=[ID_COL, TARGET])
test = test.drop(columns=[ID_COL])


train.head()


,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries
0,35.0,175.0,75.0,81.0,1.2,1.2,1.0,1.0,120.0,80.0,...,67.0,63.0,105.0,14.3,1.0,0.9,18.0,16.0,15.0,0.0
1,35.0,165.0,70.0,82.0,1.5,1.2,1.0,1.0,138.0,88.0,...,360.0,68.0,83.0,15.3,1.0,0.7,105.0,173.0,483.0,1.0
2,40.0,180.0,70.0,82.0,1.2,1.0,1.0,1.0,119.0,67.0,...,85.0,51.0,104.0,14.3,1.0,1.0,18.0,14.0,23.0,1.0
3,40.0,150.0,55.0,75.2,1.5,1.2,1.0,1.0,110.0,70.0,...,57.0,61.0,112.0,14.0,1.0,0.9,17.0,14.0,17.0,0.0
4,35.0,170.0,75.0,89.0,1.0,1.2,1.0,1.0,133.0,84.0,...,161.0,52.0,144.0,15.2,2.0,0.9,42.0,59.0,35.0,0.0


## 3. Feature Engineering

In [4]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # BMI and body composition
    df["BMI"] = df["weight(kg)"] / ((df["height(cm)"] / 100) ** 2)
    df["waist_height_ratio"] = df["waist(cm)"] / df["height(cm)"]
    df["waist_weight_ratio"] = df["waist(cm)"] / df["weight(kg)"]

    # Blood pressure
    df["pulse_pressure"] = df["systolic"] - df["relaxation"]
    df["mean_arterial_pressure"] = df["relaxation"] + (df["pulse_pressure"] / 3)

    # Lipid profile
    df["chol_hdl_ratio"] = df["Cholesterol"] / df["HDL"]
    df["ldl_hdl_ratio"] = df["LDL"] / df["HDL"]
    df["tg_hdl_ratio"] = df["triglyceride"] / df["HDL"]
    df["non_hdl_chol"] = df["Cholesterol"] - df["HDL"]

    # Liver enzymes (AST/ALT/Gtp are classic smoking/alcohol markers)
    df["ast_alt_ratio"] = df["AST"] / (df["ALT"] + 1e-3)
    df["liver_load"] = df["AST"] + df["ALT"] + df["Gtp"]
    df["log_gtp"] = np.log1p(df["Gtp"])
    df["log_ast"] = np.log1p(df["AST"])
    df["log_alt"] = np.log1p(df["ALT"])
    df["log_tg"] = np.log1p(df["triglyceride"])

    # Vision / hearing composites
    df["eyesight_avg"] = (df["eyesight(left)"] + df["eyesight(right)"]) / 2
    df["eyesight_diff"] = (df["eyesight(left)"] - df["eyesight(right)"]).abs()
    df["hearing_avg"] = (df["hearing(left)"] + df["hearing(right)"]) / 2
    df["hearing_any_impaired"] = (
        (df["hearing(left)"] > 1) | (df["hearing(right)"] > 1)
    ).astype(int)

    # Kidney function
    df["egfr_proxy"] = df["age"] * df["serum creatinine"]  # crude interaction

    # Age/hemoglobin interaction (hemoglobin differs strongly by sex/smoking)
    df["hemoglobin_age_ratio"] = df["hemoglobin"] / df["age"]

    return df


train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Replace potential inf values created by ratios (e.g. HDL == 0)
train_fe = train_fe.replace([np.inf, -np.inf], np.nan)
test_fe = test_fe.replace([np.inf, -np.inf], np.nan)
train_fe = train_fe.fillna(train_fe.median(numeric_only=True))
test_fe = test_fe.fillna(train_fe.median(numeric_only=True))

feature_cols = train_fe.columns.tolist()
print(f"Total features after engineering: {len(feature_cols)}")

X = train_fe[feature_cols].values
X_test = test_fe[feature_cols].values
y = y.values

Total features after engineering: 43


## 4. Model Parameters

In [5]:
lgb_params = dict(
    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    learning_rate=0.02,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=25,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=1.0,
    n_estimators=3000,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

xgb_params = dict(
    objective="binary:logistic",
    eval_metric="auc",
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=1.0,
    n_estimators=3000,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

cat_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    iterations=3000,
    random_seed=RANDOM_STATE,
    verbose=False,
)

## 5. Cross-Validated Training: LightGBM + XGBoost + CatBoost

In [6]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== Fold {fold + 1}/{N_SPLITS} =====")
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    # --- LightGBM ---
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    pred_lgb += model_lgb.predict_proba(X_test)[:, 1] / N_SPLITS

    # --- XGBoost ---
    model_xgb = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=150)
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    pred_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_SPLITS

    # --- CatBoost ---
    model_cat = cb.CatBoostClassifier(**cat_params)
    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150,
        verbose=False,
    )
    oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    pred_cat += model_cat.predict_proba(X_test)[:, 1] / N_SPLITS

    fold_auc_lgb = roc_auc_score(y_val, oof_lgb[val_idx])
    fold_auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
    fold_auc_cat = roc_auc_score(y_val, oof_cat[val_idx])
    print(f"Fold {fold + 1} AUC -> LGB: {fold_auc_lgb:.5f} | XGB: {fold_auc_xgb:.5f} | CAT: {fold_auc_cat:.5f}")


===== Fold 1/5 =====


c:\Users\Choy\anaconda3\envs\AIML\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 AUC -> LGB: 0.88885 | XGB: 0.88894 | CAT: 0.88893

===== Fold 2/5 =====


c:\Users\Choy\anaconda3\envs\AIML\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 AUC -> LGB: 0.88949 | XGB: 0.88882 | CAT: 0.88748

===== Fold 3/5 =====


...

Fold 3 AUC -> LGB: 0.89672 | XGB: 0.89719 | CAT: 0.89757

===== Fold 4/5 =====


c:\Users\Choy\anaconda3\envs\AIML\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 AUC -> LGB: 0.89545 | XGB: 0.89640 | CAT: 0.89616

===== Fold 5/5 =====


c:\Users\Choy\anaconda3\envs\AIML\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 AUC -> LGB: 0.88807 | XGB: 0.88813 | CAT: 0.88792


## 6. Evaluate Ensemble

In [7]:
print("===== Overall CV AUC =====")
print(f"LightGBM : {roc_auc_score(y, oof_lgb):.5f}")
print(f"XGBoost  : {roc_auc_score(y, oof_xgb):.5f}")
print(f"CatBoost : {roc_auc_score(y, oof_cat):.5f}")

oof_ensemble = (oof_lgb + oof_xgb + oof_cat) / 3
print(f"Ensemble (equal weight): {roc_auc_score(y, oof_ensemble):.5f}")

# Simple weighted grid search to optimize the blend
best_auc, best_w = 0, (1 / 3, 1 / 3, 1 / 3)
for w1 in np.arange(0, 1.01, 0.1):
    for w2 in np.arange(0, 1.01 - w1, 0.1):
        w3 = 1 - w1 - w2
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        auc = roc_auc_score(y, blend)
        if auc > best_auc:
            best_auc, best_w = auc, (w1, w2, w3)

print(f"Best blend weights (lgb, xgb, cat) = {best_w}, AUC = {best_auc:.5f}")

===== Overall CV AUC =====
LightGBM : 0.89157
XGBoost  : 0.89180
CatBoost : 0.89151
Ensemble (equal weight): 0.89271
Best blend weights (lgb, xgb, cat) = (np.float64(0.2), np.float64(0.4), np.float64(0.4)), AUC = 0.89275


## 7. Final Test Predictions & Submission File

In [8]:
w1, w2, w3 = best_w
final_pred = w1 * pred_lgb + w2 * pred_xgb + w3 * pred_cat

submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_pred})
submission.to_csv("submission.csv", index=False)
print("Saved submission to submission.csv")
submission.head()

Saved submission to submission.csv


,id,smoking
0,15000,0.810497
1,15001,0.613738
2,15002,0.464728
3,15003,0.744224
4,15004,0.912585
